# 02 - Train the Detection Head (RSNA Pneumonia)

Trains DenseNet-121 encoder + a Tiny-YOLOv2 style detection head, using the YOLO loss (coordinate + objectness + no-object + class terms).

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from torch.utils.data import DataLoader
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.rsna import (RSNADataset, load_rsna_dataframe, get_default_train_augmentation,
                                train_val_split, rsna_collate_fn)
from src.utils.visualize import show_image_boxes, show_training_curves
from src.utils.bbox_utils import decode_predictions

print("device:", config.DEVICE)


## 1. Point these at your downloaded data (see notebook 00)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


df = load_rsna_dataframe(DET_CSV_PATH)
print(len(df), "rows,", df['patientId'].nunique(), "unique images")
df.head()


In [ ]:
train_ids, val_ids = train_val_split(df, val_fraction=0.15)
print(len(train_ids), "train images,", len(val_ids), "val images")

train_ds = RSNADataset(df, train_ids, DET_IMG_PATH, augmentation=get_default_train_augmentation())
val_ds = RSNADataset(df, val_ids, DET_IMG_PATH, augmentation=None)

BATCH_SIZE = 16
# num_workers=2: DICOM decode + resize is CPU-bound; parallel workers matter
# a lot here. You may see a harmless "AssertionError: can only test a child
# process" during worker cleanup under Kaggle/Jupyter - cosmetic only,
# doesn't affect training. persistent_workers=True avoids respawning workers
# every epoch.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          persistent_workers=True, drop_last=True, collate_fn=rsna_collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        persistent_workers=True, collate_fn=rsna_collate_fn)


## 2. Peek at a training sample with its ground-truth boxes

In [ ]:
img, target, label, boxes = train_ds[next(i for i in range(len(train_ds)) if len(train_ds[i][3])>0)]
show_image_boxes(img.permute(1,2,0).numpy(), boxes, title=f"class={label}")


## 3. Build the model (detection head only) and train

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=False,
                     add_detector=True, add_segmenter=False).to(config.DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)

# Previous run: val_loss bottomed out at epoch 5 (0.34) then rose to 0.41 by
# epoch 15 - classic overfitting. Early stopping (patience=5) stops training
# once it stops improving instead of training 10 more wasted/overfit epochs;
# the checkpoint at the true best epoch is what gets kept either way.
EPOCHS = 15
history = engine.fit(
    engine.train_one_epoch_detector, engine.evaluate_detector,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt",
    monitor="loss", mode="min", scaler=scaler,
    scheduler=scheduler, patience=5,
)


## 4. Training curves & qualitative results

In [ ]:
show_training_curves({k: v for k, v in history.items() if 'loss' in k})


In [ ]:
model.load("/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt", map_location=config.DEVICE)
model.eval()

# Grid of several validation examples (matches the upstream README's
# "Detection results" example-results style: prediction=blue,
# ground truth=green, overlaid on the same panel) instead of a single
# fixed-threshold spot check.
#
# conf_threshold note: a single image at a fixed 0.3 threshold can show
# *zero* predicted boxes even when the model is working, simply because
# its confidence on that one image happens to land under the cutoff -
# especially early in training. This cell scans several images, and for
# any where nothing clears CONF_THRESHOLD it reports the highest
# confidence actually seen, so "no boxes" is diagnosable instead of a
# silent blank panel.
from src.utils.visualize import show_detection_grid

CONF_THRESHOLD = 0.15  # lowered from 0.3 for a more forgiving/diagnostic view
N_EXAMPLES = 8

positive_idxs = [i for i in range(len(val_ds)) if len(val_ds[i][3]) > 0][:N_EXAMPLES]
images, gt_boxes_list, pred_boxes_list, pred_scores_list = [], [], [], []
for idx in positive_idxs:
    img, target, label, gt_boxes = val_ds[idx]
    with torch.no_grad():
        det_out = model(img.unsqueeze(0).to(config.DEVICE))["det"][0]
    preds = decode_predictions(det_out, conf_threshold=CONF_THRESHOLD)
    if not preds:
        # nothing cleared the threshold - report what the model actually
        # output instead of silently showing an empty panel
        preds_any = decode_predictions(det_out, conf_threshold=0.0)
        max_conf = max((p["score"] for p in preds_any), default=0.0)
        print(f"[val idx {idx}] no predictions above conf_threshold={CONF_THRESHOLD} "
              f"(highest raw confidence seen: {max_conf:.3f})")
    images.append(img.permute(1, 2, 0).numpy())
    gt_boxes_list.append(gt_boxes)
    pred_boxes_list.append([p["box"] for p in preds])
    pred_scores_list.append([p["score"] for p in preds])

show_detection_grid(images, gt_boxes_list, pred_boxes_list, pred_scores_list, n_cols=4)


Checkpoint saved to `/content/detector_best.pt`. Reused in `04_Train_MTL_Joint.ipynb`.